In [21]:
import numpy as np
import pandas as pd
from pandas.core.groupby import categorical
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from xgboost import XGBRegressor
import joblib

df = pd.read_csv('../data/riyasewana_processed_ready.csv')

X = df.drop(columns=['Price','Year'])
y = df['Price']


In [22]:
categorical_col = ['Location', 'Make', 'Model', 'Gear', 'Fuel Type', 'Condition']

numerical_col = ['Mileage', 'Engine (cc)', 'Vehicle_Age']

In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_col),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_col)
    ]
)

xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ("model",XGBRegressor(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    ))
])

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Applying log1p transformation to prices...")
y_train_log = np.log1p(y_train)

print("Tranimng the XGBoost model...")
xgb_pipeline.fit(X_train, y_train_log)

y_pred_log = xgb_pipeline.predict(X_test)

y_pred = np.expm1(y_pred_log)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n--- Improved Model Evaluation ---")
print(f"Mean Absolute Error (MAE): Rs. {mae:,.0f}")
print(f"Root Mean Squared Error (RMSE): Rs. {rmse:,.0f}")
print(f"R-squared (R2): {r2:.4f}")

# 11. Export the Pipeline
joblib.dump(xgb_pipeline, '../backend/xgboost_car_price_model_log.joblib')
print("\n✅ Pipeline successfully saved to backend/xgboost_car_price_model_log.joblib")

Applying log1p transformation to prices...
Tranimng the XGBoost model...

--- Improved Model Evaluation ---
Mean Absolute Error (MAE): Rs. 878,512
Root Mean Squared Error (RMSE): Rs. 2,606,190
R-squared (R2): 0.6095

✅ Pipeline successfully saved to backend/xgboost_car_price_model_log.joblib
